# Masked Diffusion Language Model Training (v3)

This notebook trains a discrete diffusion language model following MDLM (Sahoo et al. 2024) and dLLM (Zhou 2025) on WikiText-2.

**Training Strategy:**
- Stage 1: Low noise (15% masking) for 5 epochs — learn basic reconstruction
- Stage 2: High noise (50% masking) for 5 epochs — learn harder denoising

**Target Metrics:**
- Reconstruction accuracy: >30%
- GPT-2 perplexity: <500

Once these are met, the model is ready for watermarking experiments.

## Cell 1: Environment Check

In [1]:
import sys
import torch
from pathlib import Path

# Find repo root (contains requirements.txt)
current = Path.cwd()
REPO_ROOT = None
for parent in [current] + list(current.parents):
    if (parent / "requirements.txt").exists():
        REPO_ROOT = parent
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root (requirements.txt not found)")

# Add to sys.path
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")
print(f"Python version:  {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"MPS available:   {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("✓ MPS (Apple Silicon GPU) is ready")
    device = "mps"
elif torch.cuda.is_available():
    print(f"✓ CUDA GPU available: {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("⚠ Using CPU (training will be slow)")
    device = "cpu"

print(f"\nDevice: {device}")

Repository root: /Users/idhantsingh/Desktop/diffusion-lm-watermarking
Python version:  3.11.15
PyTorch version: 2.10.0
CUDA available:  False
MPS available:   True
✓ MPS (Apple Silicon GPU) is ready

Device: mps


## Cell 2: Imports

In [2]:
import os
import json
import math
import time
import shutil
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from tqdm.auto import tqdm
from transformers import (
    BertTokenizerFast,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    get_cosine_schedule_with_warmup,
)

from models.diffusion_lm import (
    TimestepConditionedBertForMaskedLM,
    make_noised_batch,
    generate_d3pm,
    reconstruction_accuracy,
)
from data.dataset import WikiTextDataset

print("✓ All imports successful")

✓ All imports successful


## Cell 3: Configuration

In [3]:
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"

CONFIG = {
    "model_name":       "bert-base-uncased",
    "dataset_version":  "wikitext-2-raw-v1",
    "batch_size":       32,
    "max_length":       128,
    "diffusion_steps":  100,
    "min_mask_prob":    0.15,
    "log_every":        50,
    "seed":             42,
    "grad_accum_steps": 4,
    "device":           device,
    "stages": [
        {
            "name":         "stage1",
            "max_mask_prob": 0.15,
            "epochs":        5,
            "lr":            2e-5,
            "warmup":        200,
            "output_dir":    str(CHECKPOINT_DIR / "dlm-stage1"),
        },
        {
            "name":         "stage2",
            "max_mask_prob": 0.50,
            "epochs":        8,
            "lr":            1e-5,
            "warmup":        150,
            "output_dir":    str(CHECKPOINT_DIR / "dlm-stage2"),
        },
        {
            "name":         "stage3",
            "max_mask_prob": 0.75,
            "epochs":        6,
            "lr":            5e-6,
            "warmup":        100,
            "output_dir":    str(CHECKPOINT_DIR / "dlm-stage3"),
        },
    ],
}

print("═" * 60)
print("TRAINING CONFIGURATION")
print("═" * 60)
print(f"Model:              {CONFIG['model_name']}")
print(f"Dataset:            {CONFIG['dataset_version']}")
print(f"Device:             {CONFIG['device']}")
print(f"Batch size:         {CONFIG['batch_size']}")
print(f"Grad accum steps:   {CONFIG['grad_accum_steps']}")
print(f"Effective batch:    {CONFIG['batch_size'] * CONFIG['grad_accum_steps']}")
print(f"Max length:         {CONFIG['max_length']}")
print(f"Diffusion steps:    {CONFIG['diffusion_steps']}")
print(f"Seed:               {CONFIG['seed']}")
print()
print("Training stages:")
for i, stage in enumerate(CONFIG["stages"], 1):
    print(f"  Stage {i}: {stage['name']}")
    print(f"    Max mask prob:  {stage['max_mask_prob']:.2f}")
    print(f"    Epochs:         {stage['epochs']}")
    print(f"    Learning rate:  {stage['lr']:.0e}")
    print(f"    Warmup steps:   {stage['warmup']}")
    print(f"    Output dir:     {stage['output_dir']}")
    print()
print("═" * 60)

════════════════════════════════════════════════════════════
TRAINING CONFIGURATION
════════════════════════════════════════════════════════════
Model:              bert-base-uncased
Dataset:            wikitext-2-raw-v1
Device:             mps
Batch size:         32
Grad accum steps:   4
Effective batch:    128
Max length:         128
Diffusion steps:    100
Seed:               42

Training stages:
  Stage 1: stage1
    Max mask prob:  0.15
    Epochs:         5
    Learning rate:  2e-05
    Warmup steps:   200
    Output dir:     /Users/idhantsingh/Desktop/diffusion-lm-watermarking/checkpoints/dlm-stage1

  Stage 2: stage2
    Max mask prob:  0.50
    Epochs:         8
    Learning rate:  1e-05
    Warmup steps:   150
    Output dir:     /Users/idhantsingh/Desktop/diffusion-lm-watermarking/checkpoints/dlm-stage2

  Stage 3: stage3
    Max mask prob:  0.75
    Epochs:         6
    Learning rate:  5e-06
    Warmup steps:   100
    Output dir:     /Users/idhantsingh/Desktop/diffusion-l

## Cell 4: Checkpoint Helpers

In [4]:
def save_checkpoint(model, tokenizer, save_dir: str) -> None:
    """Save model and tokenizer to directory."""
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)


def save_resume_state(state: dict, output_dir: str) -> None:
    """Atomically save resume state to JSON."""
    path = os.path.join(output_dir, "resume_state.json")
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f, indent=2)
    os.replace(tmp, path)


def load_resume_state(output_dir: str) -> dict | None:
    """Load resume state from JSON, or None if doesn't exist."""
    path = os.path.join(output_dir, "resume_state.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def save_epoch_checkpoint(
    model, tokenizer, output_dir: str, stage_name: str, epoch: int
) -> str:
    """Save a named per-epoch checkpoint for watermarking experiments."""
    epoch_dir = os.path.join(
        output_dir, "epoch_ckpts", f"{stage_name}_epoch{epoch}"
    )
    save_checkpoint(model, tokenizer, epoch_dir)
    return epoch_dir


print("✓ Checkpoint helpers loaded")

✓ Checkpoint helpers loaded


## Cell 5: GPT-2 Perplexity Scorer

In [5]:
# Cache GPT-2 globally so it only loads once
_gpt2_model = None
_gpt2_tok = None


def get_gpt2_scorer(device: str):
    """Lazy-load GPT-2 scorer (cached globally)."""
    global _gpt2_model, _gpt2_tok
    if _gpt2_model is None:
        print("Loading GPT-2 scorer...")
        _gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device).eval()
        _gpt2_tok = GPT2TokenizerFast.from_pretrained("gpt2")
        _gpt2_tok.pad_token = _gpt2_tok.eos_token
        print("✓ GPT-2 scorer ready")
    return _gpt2_model, _gpt2_tok


def score_perplexity(text: str, device: str) -> float:
    """Score text perplexity using GPT-2."""
    gpt2, tok = get_gpt2_scorer(device)
    enc = tok(
        text, return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    if enc["input_ids"].shape[1] < 2:
        return float("inf")
    with torch.no_grad():
        loss = gpt2(**enc, labels=enc["input_ids"]).loss
    return math.exp(loss.item())


print("✓ GPT-2 scorer functions loaded")

✓ GPT-2 scorer functions loaded


## Cell 6: Evaluation Function

In [6]:
def evaluate(
    model, tokenizer, val_dataset, device: str, n_recon: int = 50, n_gen: int = 5
) -> dict:
    """
    Run two evaluations:
    1. Reconstruction accuracy — can model fill in masked tokens?
    2. GPT-2 perplexity — how fluent are generated samples?

    Returns dict with accuracy, mean_perplexity, and sample texts.
    Target: accuracy > 30%, perplexity < 500 to proceed to watermarking.
    """
    model.eval()

    # Reconstruction accuracy
    sentences = [
        val_dataset.texts[i] for i in range(min(n_recon, len(val_dataset)))
    ]
    recon = reconstruction_accuracy(
        model,
        tokenizer,
        sentences,
        mask_prob=0.15,
        steps=CONFIG["diffusion_steps"],
        device=device,
    )

    # GPT-2 perplexity on generated samples
    samples = []
    ppls = []
    for _ in range(n_gen):
        text = generate_d3pm(
            model,
            tokenizer,
            length=64,
            steps=CONFIG["diffusion_steps"],
            min_mask_prob=CONFIG["min_mask_prob"],
            max_mask_prob=0.50,
            device=device,
            temperature=1.0,
            top_k=50,
        )
        samples.append(text)
        ppls.append(score_perplexity(text, device))

    finite = [p for p in ppls if p != float("inf")]
    mean_ppl = sum(finite) / len(finite) if finite else float("inf")

    print(f"  Reconstruction accuracy: {recon['accuracy']*100:.1f}%  "
          f"({recon['correct']}/{recon['total']} tokens)")
    print(f"  Mean perplexity: {mean_ppl:.1f}")
    print(f"  Samples:")
    for i, s in enumerate(samples):
        print(f"    [{i+1}] {s}")

    model.train()
    return {
        "accuracy": recon["accuracy"],
        "correct": recon["correct"],
        "total": recon["total"],
        "mean_perplexity": mean_ppl,
        "samples": samples,
    }


print("✓ Evaluation function loaded")

✓ Evaluation function loaded


## Cell 7: Data Loading

In [7]:
torch.manual_seed(CONFIG["seed"])

tokenizer = BertTokenizerFast.from_pretrained(CONFIG["model_name"])

train_dataset = WikiTextDataset(
    split="train",
    max_length=CONFIG["max_length"],
    dataset_version=CONFIG["dataset_version"],
)

val_dataset = WikiTextDataset(
    split="validation",
    max_length=CONFIG["max_length"],
    dataset_version=CONFIG["dataset_version"],
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=0,
)

print(f"\nTrain size:           {len(train_dataset)} samples")
print(f"Validation size:      {len(val_dataset)} samples")
print(f"Batches per epoch:    {len(train_loader)}")
print(f"Updates per epoch:    {len(train_loader) // CONFIG['grad_accum_steps']}")

📝 WikiText-2 train: 16047 samples after cleaning (removed 6781 noisy samples)
📝 WikiText-2 validation: 1713 samples after cleaning (removed 667 noisy samples)

Train size:           16047 samples
Validation size:      1713 samples
Batches per epoch:    501
Updates per epoch:    125


## Cell 8: Model Initialization + Resume Logic

In [8]:
# Check if we can resume from the last stage
last_stage = CONFIG["stages"][-1]
resume_state = load_resume_state(last_stage["output_dir"])

# Force training from scratch for cleaned-data runs
FORCE_FRESH_START = True

if FORCE_FRESH_START:
    print("FORCE_FRESH_START=True — ignoring resume checkpoints")
    resume_state = None

if resume_state is not None:
    print("Found resume_state.json — resuming training")
    print(f"  Stage {resume_state['start_stage_idx'] + 1}, "
          f"Epoch {resume_state['start_epoch_idx'] + 1}")
    
    # Load model from latest checkpoint
    latest_dir = os.path.join(last_stage["output_dir"], "latest")
    model = TimestepConditionedBertForMaskedLM.from_pretrained(
        latest_dir,
        num_steps=CONFIG["diffusion_steps"],
        device=CONFIG["device"],
    )
    print(f"  Loaded from: {latest_dir}")
    
    # Restore global state
    global_state = {
        "start_stage_idx": resume_state["start_stage_idx"],
        "start_epoch_idx": resume_state["start_epoch_idx"],
        "best_loss": resume_state["best_loss"],
        "best_epoch": resume_state["best_epoch"],
        "best_ckpt_dir": resume_state["best_ckpt_dir"],
    }
    print(f"  Best loss so far: {global_state['best_loss']:.4f}")
    
else:
    V5_BEST = str(CHECKPOINT_DIR / "dlm-stage1" / "best")
    print(f"No checkpoint found at {V5_BEST}")
    print(f"Starting fresh from {CONFIG['model_name']}")
    model = TimestepConditionedBertForMaskedLM.from_pretrained(
        CONFIG["model_name"],
        num_steps=CONFIG["diffusion_steps"],
        device=CONFIG["device"],
    )
    # Freeze embeddings only on completely fresh start
    print("Freezing BERT embeddings for fresh training...")
    for param in model.base.bert.embeddings.parameters():
        param.requires_grad = False
    
    global_state = {
        "start_stage_idx": 0,
        "start_epoch_idx": 0,
        "best_loss":       float("inf"),
        "best_epoch":      None,
        "best_ckpt_dir":   None,
    }

model.train()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nModel parameters:")
print(f"  Total:      {total_params:,}")
print(f"  Trainable:  {trainable_params:,}")
print(f"  Frozen:     {frozen_params:,}")
print(f"\n✓ Model ready on {CONFIG['device']}")


FORCE_FRESH_START=True — ignoring resume checkpoints
No checkpoint found at /Users/idhantsingh/Desktop/diffusion-lm-watermarking/checkpoints/dlm-stage1/best
Starting fresh from bert-base-uncased


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Freezing BERT embeddings for fresh training...

Model parameters:
  Total:      109,593,402
  Trainable:  85,756,218
  Frozen:     23,837,184

✓ Model ready on mps


## Cell 9: Training Loop

**What happens here:**

For each training stage:
1. **Optimizer & Scheduler**: Fresh optimizer for each stage with stage-specific learning rate
2. **Forward Process**: Random tokens are masked at rate `max_mask_prob` (15% → 50%)
3. **Model Prediction**: Model predicts original tokens from masked input
4. **Gradient Accumulation**: Loss divided by 4, accumulated over 4 batches = effective batch 128
5. **Checkpointing**:
   - `latest/` — Resume point if interrupted
   - `best/` — Lowest loss checkpoint
   - `epoch_ckpts/` — Per-epoch checkpoints for watermarking experiments
6. **Resume State**: JSON file tracks progress (stage, epoch, best loss)

**Why 2 stages?**
- Stage 1 (15% masking): Learn basic token reconstruction (easier task)
- Stage 2 (50% masking): Learn harder denoising (generalization)

**Progress bars:**
- Outer: Epochs within current stage
- Inner: Batches within current epoch

In [9]:
try:
    print("\n" + "═" * 80)
    print("STARTING TRAINING")
    print("═" * 80 + "\n")
    
    for stage_idx in range(global_state["start_stage_idx"], len(CONFIG["stages"])):
        stage = CONFIG["stages"][stage_idx]
        
        print(f"\n{'─' * 80}")
        print(f"STAGE {stage_idx + 1}: {stage['name'].upper()}")
        print(f"{'─' * 80}")
        print(f"Max mask probability: {stage['max_mask_prob']:.2f}")
        print(f"Epochs:               {stage['epochs']}")
        print(f"Learning rate:        {stage['lr']:.0e}")
        print(f"Warmup steps:         {stage['warmup']}")
        print(f"Output directory:     {stage['output_dir']}")
        print(f"{'─' * 80}\n")
        
        # Fresh optimizer and scheduler for this stage
        # WHY: Each stage has different learning rates and noise levels
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=stage["lr"],
            weight_decay=0.01,
        )
        
        total_steps = stage["epochs"] * len(train_loader)
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=stage["warmup"],
            num_training_steps=total_steps,
        )
        
        # Determine starting epoch for this stage
        start_epoch = (
            global_state["start_epoch_idx"]
            if stage_idx == global_state["start_stage_idx"]
            else 0
        )
        
        # Epoch progress bar
        epoch_pbar = tqdm(
            range(start_epoch, stage["epochs"]),
            desc=f"Stage {stage_idx+1} Epochs",
            initial=start_epoch,
            total=stage["epochs"],
            position=0,
        )
        
        for epoch in epoch_pbar:
            epoch_start = time.time()
            running_loss = 0.0
            log_window_loss = 0.0
            log_window_count = 0
            
            # Batch progress bar
            batch_pbar = tqdm(
                enumerate(train_loader, start=1),
                total=len(train_loader),
                desc=f"  Epoch {epoch+1}/{stage['epochs']}",
                position=1,
                leave=False,
            )
            
            for batch_idx, input_ids in batch_pbar:
                input_ids = input_ids.to(CONFIG["device"])
                
                # STEP 1: Forward diffusion process
                # Randomly mask tokens at rate stage['max_mask_prob']
                # This simulates the corruption process q(x_t | x_0)
                x_t, labels, t = make_noised_batch(
                    input_ids,
                    tokenizer=tokenizer,
                    steps=CONFIG["diffusion_steps"],
                    min_mask_prob=CONFIG["min_mask_prob"],
                    max_mask_prob=stage["max_mask_prob"],
                    device=CONFIG["device"],
                )
                
                # STEP 2: Model prediction (reverse process)
                # Model learns to predict original tokens from masked input
                # This trains the denoiser p_θ(x_0 | x_t, t)
                attention_mask = input_ids.ne(tokenizer.pad_token_id).long()
                out = model(input_ids=x_t, t=t, attention_mask=attention_mask, labels=labels)
                loss = out["loss"]
                
                # STEP 3: Gradient accumulation
                # WHY: Accumulate gradients over 4 batches to simulate batch size of 128
                # (32 * 4 = 128) without running out of GPU memory
                loss = loss / CONFIG["grad_accum_steps"]
                loss.backward()
                
                # Accumulate loss (multiply back for correct display)
                actual_loss = loss.item() * CONFIG["grad_accum_steps"]
                running_loss += actual_loss
                log_window_loss += actual_loss
                log_window_count += 1
                
                # STEP 4: Optimizer step (every grad_accum_steps batches)
                if batch_idx % CONFIG["grad_accum_steps"] == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                
                # Update batch progress bar
                if log_window_count > 0:
                    avg_loss = log_window_loss / log_window_count
                    batch_pbar.set_postfix(
                        loss=f"{avg_loss:.4f}",
                        lr=f"{scheduler.get_last_lr()[0]:.2e}"
                    )
                
                # Text logging (less frequent)
                if CONFIG["log_every"] and batch_idx % CONFIG["log_every"] == 0:
                    avg_loss = log_window_loss / log_window_count
                    current_lr = scheduler.get_last_lr()[0]
                    tqdm.write(
                        f"    stage={stage_idx+1} "
                        f"epoch={epoch+1}/{stage['epochs']} "
                        f"step={batch_idx}/{len(train_loader)} "
                        f"loss={avg_loss:.4f} "
                        f"lr={current_lr:.2e}"
                    )
                    log_window_loss = 0.0
                    log_window_count = 0
            
            batch_pbar.close()
            
            # STEP 5: Epoch complete — save checkpoints
            epoch_avg_loss = running_loss / len(train_loader)
            epoch_time = time.time() - epoch_start
            
            # Save latest checkpoint (for resuming if interrupted)
            latest_dir = os.path.join(stage["output_dir"], "latest")
            save_checkpoint(model, tokenizer, latest_dir)
            
            # STEP 6: Save resume state (atomic write via tmp file)
            # WHY: If training is interrupted, we can resume from this exact point
            resume_data = {
                "start_stage_idx": stage_idx,
                "start_epoch_idx": epoch + 1,  # Next epoch to run
                "best_loss": global_state["best_loss"],
                "best_epoch": global_state["best_epoch"],
                "best_ckpt_dir": global_state["best_ckpt_dir"],
            }
            save_resume_state(resume_data, stage["output_dir"])
            
            # STEP 7: Track best checkpoint
            is_best = False
            if epoch_avg_loss < global_state["best_loss"]:
                global_state["best_loss"] = epoch_avg_loss
                global_state["best_epoch"] = f"{stage['name']}_epoch{epoch+1}"
                best_dir = os.path.join(stage["output_dir"], "best")
                global_state["best_ckpt_dir"] = best_dir
                save_checkpoint(model, tokenizer, best_dir)
                is_best = True
                
                # Update resume state with new best
                resume_data["best_loss"] = global_state["best_loss"]
                resume_data["best_epoch"] = global_state["best_epoch"]
                resume_data["best_ckpt_dir"] = global_state["best_ckpt_dir"]
                save_resume_state(resume_data, stage["output_dir"])
            
            # STEP 8: Save per-epoch checkpoint
            # WHY: Needed for watermarking experiments — we'll compare
            # watermarked vs non-watermarked checkpoints from same epoch
            epoch_ckpt_dir = save_epoch_checkpoint(
                model, tokenizer, stage["output_dir"], stage["name"], epoch + 1
            )
            
            # Update epoch progress bar
            epoch_pbar.set_postfix(
                loss=f"{epoch_avg_loss:.4f}",
                time=f"{epoch_time:.0f}s",
                best="★" if is_best else ""
            )
            
            # Print epoch summary
            tqdm.write(
                f"\nEpoch {epoch+1}/{stage['epochs']} complete: "
                f"avg_loss={epoch_avg_loss:.4f} "
                f"time={epoch_time:.1f}s"
            )
            if is_best:
                tqdm.write(f"★ New best! loss={epoch_avg_loss:.4f}")
            tqdm.write(f"Saved: {epoch_ckpt_dir}\n")
        
        epoch_pbar.close()
        
        # Stage complete — run evaluation
        print(f"\n{'─' * 80}")
        print(f"STAGE {stage_idx + 1} COMPLETE — EVALUATION")
        print(f"{'─' * 80}\n")
        
        eval_results = evaluate(
            model, tokenizer, val_dataset, device=CONFIG["device"], n_recon=200, n_gen=20
        )
        
        print(f"\nStage {stage_idx + 1} summary:")
        print(f"  Reconstruction accuracy: {eval_results['accuracy']*100:.1f}% "
              f"(target: >30%)")
        print(f"  Mean perplexity:         {eval_results['mean_perplexity']:.1f} "
              f"(target: <500)")
        
        # Check if targets met
        if eval_results["accuracy"] > 0.30 and eval_results["mean_perplexity"] < 500:
            print("  ✓ Targets met! Model is ready for watermarking experiments.")
        else:
            print("  ⚠ Targets not yet met. Continue training.")
        
        print()
        
        # Unfreeze embeddings after Stage 1 for later stages
        if stage_idx == 0:
            print("Unfreezing BERT embeddings for stages 2 and 3...")
            for param in model.base.bert.embeddings.parameters():
                param.requires_grad = True
            trainable = sum(p.numel() for p in model.parameters() 
                           if p.requires_grad)
            print(f"Trainable params now: {trainable:,}")
        
        # Reset for next stage
        global_state["start_epoch_idx"] = 0
    
    # All stages complete
    print("\n" + "═" * 80)
    print("TRAINING COMPLETE")
    print("═" * 80 + "\n")
    print(f"Best loss:       {global_state['best_loss']:.4f}")
    print(f"Best epoch:      {global_state['best_epoch']}")
    print(f"Best checkpoint: {global_state['best_ckpt_dir']}")
    print()
    
    # Final evaluation on best checkpoint
    print("Running final evaluation on best checkpoint...")
    best_model = TimestepConditionedBertForMaskedLM.from_pretrained(
        global_state["best_ckpt_dir"],
        num_steps=CONFIG["diffusion_steps"],
        device=CONFIG["device"],
    )
    final_results = evaluate(
        best_model, tokenizer, val_dataset, device=CONFIG["device"], n_recon=200, n_gen=20
    )
    print(f"\nFinal results:")
    print(f"  Accuracy:    {final_results['accuracy']*100:.1f}%")
    print(f"  Perplexity:  {final_results['mean_perplexity']:.1f}")
    print()
    
except KeyboardInterrupt:
    print("\n\n" + "!" * 80)
    print("TRAINING INTERRUPTED")
    print("!" * 80 + "\n")
    
    # Save current state
    current_stage = CONFIG["stages"][global_state["start_stage_idx"]]
    latest_dir = os.path.join(current_stage["output_dir"], "latest")
    save_checkpoint(model, tokenizer, latest_dir)
    
    resume_data = {
        "start_stage_idx": global_state["start_stage_idx"],
        "start_epoch_idx": global_state["start_epoch_idx"],
        "best_loss": global_state["best_loss"],
        "best_epoch": global_state["best_epoch"],
        "best_ckpt_dir": global_state["best_ckpt_dir"],
    }
    save_resume_state(resume_data, current_stage["output_dir"])
    
    print(f"Saved latest checkpoint to: {latest_dir}")
    print(f"Saved resume state to: {current_stage['output_dir']}/resume_state.json")
    print("\n✓ Safe to stop. Re-run Cell 8 and Cell 9 to resume from last completed epoch.")
    print()



════════════════════════════════════════════════════════════════════════════════
STARTING TRAINING
════════════════════════════════════════════════════════════════════════════════


────────────────────────────────────────────────────────────────────────────────
STAGE 1: STAGE1
────────────────────────────────────────────────────────────────────────────────
Max mask probability: 0.15
Epochs:               5
Learning rate:        2e-05
Warmup steps:         200
Output directory:     /Users/idhantsingh/Desktop/diffusion-lm-watermarking/checkpoints/dlm-stage1
────────────────────────────────────────────────────────────────────────────────



Stage 1 Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

  Epoch 1/5:   0%|          | 0/501 [00:00<?, ?it/s]

    stage=1 epoch=1/5 step=50/501 loss=2.3506 lr=1.20e-06
    stage=1 epoch=1/5 step=100/501 loss=2.3543 lr=2.50e-06
    stage=1 epoch=1/5 step=150/501 loss=2.2919 lr=3.70e-06
    stage=1 epoch=1/5 step=200/501 loss=2.2499 lr=5.00e-06
    stage=1 epoch=1/5 step=250/501 loss=2.2067 lr=6.20e-06
    stage=1 epoch=1/5 step=300/501 loss=2.1565 lr=7.50e-06
    stage=1 epoch=1/5 step=350/501 loss=2.1664 lr=8.70e-06
    stage=1 epoch=1/5 step=400/501 loss=2.1092 lr=1.00e-05
    stage=1 epoch=1/5 step=450/501 loss=2.1067 lr=1.12e-05
    stage=1 epoch=1/5 step=500/501 loss=2.1046 lr=1.25e-05


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch 6/6 complete: avg_loss=3.8488 time=939.8s
Saved: /Users/idhantsingh/Desktop/diffusion-lm-watermarking/checkpoints/dlm-stage3/epoch_ckpts/stage3_epoch6


────────────────────────────────────────────────────────────────────────────────
STAGE 3 COMPLETE — EVALUATION
────────────────────────────────────────────────────────────────────────────────

  Reconstruction accuracy: 65.6%  (1681/2561 tokens)
  Mean perplexity: 109.1
  Samples:
    [1] aon is also the name for the entity known as the brain which drives the theru ' s true creation. since the etymology derives from a pun of the word " given the origin of the the. in its own domain, the letter may also be used by and by another that has assumed that name "
    [2] in was competing in the olympics, she set a record and set a record for the first leg of his world record beating the world in the 100 400 metres. she broke his favourite at the year which making her ' first best of ' winning this event. she record was set of records f

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  Reconstruction accuracy: 64.4%  (1636/2540 tokens)
  Mean perplexity: 175.8
  Samples:
    [1] the v. 2 ( 1969 - 1971 ). his or here? the city and a woman, a city he exists, and it exists. and he walks by it. she " a an river, which is here, not, by the imaginations of a " woman of youth "... an angel.
    [2] ), ( 2001 ) " the art of... " (. video 2001 ) - the video that she was in is " the art of the.. ". later on ), ( the louvre in france ( 2002 ) " a to the were played by her had to the " p. " ".
    [3] tracks 3 " n. " features the addition of -, and the 6 " s is the end and.... the tracks were featured in robyn ' s 1997 p. i. s album, the complete collection. " an angel is a " 7 " 7 " single from the original cd ( 1978 ) for
    [4] " m., in bed with my aunt - - " m ) ". i mean be sure. she, i., at her ( the " of ) me as therein ), not, as that, ' s no relation, not only ", " but more of "even, at
    [5] ( " after the heart ). and they were distinct from the language : '.. to 

## Cell 10: Watermarking Evaluation Protocol
Freeze the canonical base model and define a reproducible protocol for all baseline vs watermark comparisons.

In [17]:
# Random-seed sweep (5 seeds) across stage best checkpoints
# Runs a controlled comparison with identical protocol per checkpoint.

import random
import statistics
import numpy as np


def _set_eval_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def run_random_seed_sweep(num_seeds: int = 5, seed_pool=(100, 10000)):
    stage_ckpts = {
        "stage1_best": str(CHECKPOINT_DIR / "dlm-stage1" / "best"),
        "stage2_best": str(CHECKPOINT_DIR / "dlm-stage2" / "best"),
        "stage3_best": str(CHECKPOINT_DIR / "dlm-stage3" / "best"),
    }

    # Sample random seeds once per run and keep them fixed for all checkpoints
    random_seeds = sorted(random.sample(range(seed_pool[0], seed_pool[1]), num_seeds))
    print(f"Random seeds for this run: {random_seeds}")

    rows = []
    for stage_name, ckpt_dir in stage_ckpts.items():
        if not os.path.exists(ckpt_dir):
            print(f"WARNING: Missing checkpoint for {stage_name}: {ckpt_dir}")
            continue

        print(f"\nEvaluating {stage_name}: {ckpt_dir}")
        accs = []
        ppls = []

        for s in random_seeds:
            _set_eval_seed(s)
            model = TimestepConditionedBertForMaskedLM.from_pretrained(
                ckpt_dir,
                num_steps=CONFIG["diffusion_steps"],
                device=CONFIG["device"],
            )
            out = evaluate(
                model,
                tokenizer,
                val_dataset,
                device=CONFIG["device"],
                n_recon=EVAL_PROTOCOL["n_recon"],
                n_gen=EVAL_PROTOCOL["n_gen"],
            )
            accs.append(out["accuracy"])
            ppls.append(out["mean_perplexity"])

        acc_mean = statistics.mean(accs)
        acc_std = statistics.stdev(accs) if len(accs) > 1 else 0.0
        ppl_mean = statistics.mean(ppls)
        ppl_std = statistics.stdev(ppls) if len(ppls) > 1 else 0.0

        rows.append((stage_name, acc_mean, acc_std, ppl_mean, ppl_std))

    if not rows:
        print("No stages were evaluated. Check checkpoint paths.")
        return

    print("\n" + "=" * 92)
    print("RANDOM-SEED SWEEP SUMMARY (mean +- std)")
    print("=" * 92)
    print(f"{'Checkpoint':<14} | {'Recon accuracy (mean +- std)':<34} | {'GPT-2 PPL (mean +- std)':<30}")
    print("-" * 92)
    for name, am, asd, pm, psd in rows:
        acc_txt = f"{am*100:.2f}% +- {asd*100:.2f}%"
        ppl_txt = f"{pm:.2f} +- {psd:.2f}"
        print(f"{name:<14} | {acc_txt:<34} | {ppl_txt:<30}")
    print("=" * 92)


# Run sweep
run_random_seed_sweep(num_seeds=5)


Random seeds for this run: [1258, 2828, 5035, 9048, 9249]

Evaluating stage1_best: /Users/idhantsingh/Desktop/diffusion-lm-watermarking/checkpoints/dlm-stage1/best


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  Reconstruction accuracy: 62.1%  (1592/2565 tokens)
  Mean perplexity: 148.5
  Samples:
    [1] ( and his ( " she ). " he does as both in the film, mr., and he, and in ( 2010 ), it by, mr. r. as in the song, with that one with the song, she, that he is. she ( is ( he ), )
    [2] ( i. 11 and 1. m. ( a. " i ) iii ).,. ii. i. of the 1st. " b. 1 = 2, ( 7 i. ) ) - 8 ), john on him on. on 8 october / 13. ( i i 13 ].
    [3] ( s ) ; and ( s ) indicate a number. example. used in greek : : " s / ". or : " - i on the ones " from ( s ) equivalents. " v. x ". or i - of - s, " b ' : " for the i
    [4] of the little and the whole and, they are carried by : the,, and it - tos of,, to, who and who ( the " little people in the world ", and of my the ( books ) and as, the his - like, and the don ' s and ladies of
    [5] the to subsequent to this, the first scenes have changed by itself into a good time, as and some of the frescoes in the last supper of the last century were kept at the church from t

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  Reconstruction accuracy: 63.4%  (1650/2601 tokens)
  Mean perplexity: 162.6
  Samples:
    [1] we both think, the names of thats refer to all four kinds of distinction, the ' the d. ' ', ' r. and ' ( on (, on. ) ) ". " in the sense that ( it was r. ) ' " the d., is r, and the
    [2] and he and to ( r. ) ( m ) a., you, g. ( j., a.,. ). the latter of which m. with him, ( m. d. ) b. ( e ) s. / l ) is the inhabitants of thurn dresden, germany
    [3] once, " he did well the thing ", a she and i was ' " ( " the mother and she of her ', he ' who was her as well " ). he had a she was to the day for the next,, then a man ' s night ( the right the ) "
    [4] i was in it. - " and the end. from this song from her album i and the rain : " 1 ) ". " in the rain, " " 1 ", on the left ep. the track " 2, " " the out of us. " i ) " from which " it
    [5] on the cover, i noticed " he could live found the " home of the " in the city today,. i wrote my name. with the words " in seattle ", to me, it, t

...

  Reconstruction accuracy: 65.3%  (1682/2576 tokens)
  Mean perplexity: 136.8
  Samples:
    [1] jimmy and the machine of 2009 features the released of an album, first as the soundtrack for music from the end of 1982, and, recording for the live album and later by the smiths ' the music compilation album,, band and 1998 ( " classic years " ), the album by one man band ;
    [2] " always has been a time that a lesbian just take away from the. ", she said that she has no choice. who would she lie about her truth story, with the exception of ' saying ' as ' but what she is able to find the truth in the story is a man ' s tale '
    [3] and she returned in 1987 for the single singles third - single the.. the. : or, etc., etc., from that album,.. ". the.., " from the album, in conjunction with " lite " to her. : the. : that person "..
    [4] peter howard was appointed to cabinet as finance minister, the vice - minister for transport for information ; minister for s secretary, and subsequen

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  Reconstruction accuracy: 64.2%  (1633/2544 tokens)
  Mean perplexity: 138.3
  Samples:
    [1] " intro ". " '., '. " " " back to the original " " " ', " on it ' live. " ' now! " ], " " ' " at the rock ". ", ". ' " *. " " on the ground. '. " "
    [2] for a time, however, she was been seen in line with other characters, as to what she was looking in the. well " she was certain she looked good at. the actress stated that her character at the shootings, the times she was seen with the actress, and her face was very thin.
    [3] batchelder had brief appearances as himself, such as " aniston " the day ", a result of a number of controversy over the film ' s story as to how he would appear in the principal role. wither success, did not permanently appear in the credits of the time of the production.
    [4] several other songs in the album continue to do with the songwriting ' s original works, although got to be young " is the first of a lot of recorded and written, a collection of carey

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  Reconstruction accuracy: 63.9%  (1655/2588 tokens)
  Mean perplexity: 129.8
  Samples:
    [1] denys were sent to the scene, following being victim to a policeman. st. lucia was aware that shots had been fired at one of the women. he was on guard after being killed while the attack. firing at the same time, the returned polices on the following day and two years later ).
    [2] he served in the u. s. army ' s headquarters, united states during the war and was later employed by the iraqi army, when the general served in the army ' s artillery, in the third coalition air - air force, and in the army air corps division, which served the coalition, usa.
    [3] an electric or acoustic bass is the most common for example, and even the same it it can be used to other media used, like music, in the bird ' s call, musical instruments, or songs instruments ( ( instruments in musicians use it ). for the same purpose, their can have a different pitch
    [4] a medieval church exists in the vil

## Cell 11: Standalone Evaluation

Run this cell independently any time to evaluate the current best checkpoint.

In [18]:
# Run this cell any time independently to check current model quality

EVAL_CKPT = str(CHECKPOINT_DIR / "dlm-stage3" / "best")

if os.path.exists(EVAL_CKPT):
    print(f"Loading checkpoint: {EVAL_CKPT}\n")
    
    eval_model = TimestepConditionedBertForMaskedLM.from_pretrained(
        EVAL_CKPT,
        num_steps=CONFIG["diffusion_steps"],
        device=CONFIG["device"],
    )
    
    print("Evaluating best checkpoint...\n")
    results = evaluate(
        eval_model,
        tokenizer,
        val_dataset,
        device=CONFIG["device"],
        n_recon=200,
        n_gen=20,
    )
    
    print(f"\n{'═' * 60}")
    print("EVALUATION SUMMARY")
    print(f"{'═' * 60}")
    print(f"Reconstruction accuracy:  {results['accuracy']*100:.1f}%  (target: >30%)")
    print(f"Mean GPT-2 perplexity:    {results['mean_perplexity']:.1f}  (target: <500)")
    
    if results["accuracy"] > 0.30 and results["mean_perplexity"] < 500:
        print("\n✓ Model is ready for watermarking experiments!")
    else:
        print("\n⚠ Continue training to meet target metrics.")
    
    print(f"{'═' * 60}")
    
else:
    print(f"No checkpoint found at {EVAL_CKPT}")
    print("Run Cell 9 first to train the model.")

Loading checkpoint: /Users/idhantsingh/Desktop/diffusion-lm-watermarking/checkpoints/dlm-stage3/best



Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Evaluating best checkpoint...

  Reconstruction accuracy: 64.6%  (1664/2577 tokens)
  Mean perplexity: 124.7
  Samples:
    [1] unlike earlier songs, dancing in the summer ends with lyrics by bess, who usually begins with a hymn title : " john ' s kinder got a child ". he describes the beginning so that " a single mother " take him to sleep at her home. " his song " in the dark " "
    [2] " a ", it is two instances of the of the present. " the two all of the instances of not being. and the way the notebooks are to a, and is and and the adjectives, and and both thevocations and a conversation, that are part of the,. ".
    [3] robert found his wife are young by his death, but on the 29th she was named by his father, ( " the widow or the lord it ' s a day mother " ), after him, he took his family to the old plantation, and the home for the family at lawrence ' s old plantation.
    [4] a u. s. court, gis, or the department of ellis, j., is responsible for the collection of data in the c